In [ ]:
from aicspylibczi import CziFile

# Load CZI
czi = CziFile("stack_test.czi")

# Meta is already an ElementTree Element
root = czi.meta

# --- Get Objective Magnification ---
objective = root.find(".//Objectives/Objective")

if objective is not None:
    # Extract relevant info
    lens_id = objective.attrib.get("Id")
    lens_name = objective.attrib.get("Name")
    magnification = objective.findtext("NominalMagnification")
    na = objective.findtext("LensNA")
    wd = objective.findtext("WorkingDistance")
    pupil = objective.findtext("PupilGeometry")
    immersion = objective.findtext("Immersion")
    immersion_ri = objective.findtext("ImmersionRefractiveIndex")
    model = objective.find("Manufacturer/Model").text if objective.find("Manufacturer/Model") is not None else None

    print("🧪 Objective Metadata:")
    print(f"  ID: {lens_id}")
    print(f"  Name: {lens_name}")
    print(f"  Magnification: {magnification}x")
    print(f"  Numerical Aperture (NA): {na}")
    print(f"  Working Distance: {wd} µm")
    print(f"  Pupil Geometry: {pupil}")
    print(f"  Immersion Type: {immersion}")
    print(f"  Immersion Refractive Index: {immersion_ri}")
    print(f"  Manufacturer Model: {model}")
else:
    print("No objective metadata found.")

# --- Get Pixel Size in X/Y/Z ---
scaling_items = root.find(".//Scaling/Items")

# Dictionary to hold pixel sizes
pixel_sizes = {}

# Loop over each Distance element (X, Y, Z)
for distance in scaling_items.findall("Distance"):
    axis = distance.attrib["Id"]
    value = float(distance.findtext("Value"))
    units = distance.findtext("DefaultUnitFormat")
    pixel_sizes[axis] = (value, units)

# Print nicely
print("🧬 Pixel Size Metadata:")
for axis in ["X", "Y", "Z"]:
    val, unit = pixel_sizes.get(axis, (None, None))
    if val:
        print(f"  {axis}: {val:.3e} {unit}")
    else:
        print(f"  {axis}: not found")

# --- Get Image Dimensions (X, Y, Z) ---
shape = czi.get_dims_shape()
dims, shape_values = shape[0], shape[1]

# Convert to dictionary for readability
dims_dict = dict(zip(dims, shape_values))

# Extract the relevant dimensions if they exist
size_x = dims_dict.get("X", None)
size_y = dims_dict.get("Y", None)
size_z = dims_dict.get("Z", None)

print("\n📏 Image Dimensions (in pixels):")
print(f"  X: {size_x}")
print(f"  Y: {size_y}")
print(f"  Z: {size_z}")

# --- Compute total physical size ---
if all(axis in pixel_sizes for axis in ["X", "Y"]) and (size_x and size_y):
    size_x_um = size_x * pixel_sizes["X"][0]
    size_y_um = size_y * pixel_sizes["Y"][0]
    print("\n🌍 Physical Field of View:")
    print(f"  X: {size_x_um:.2f} {pixel_sizes['X'][1]}")
    print(f"  Y: {size_y_um:.2f} {pixel_sizes['Y'][1]}")
    if size_z and "Z" in pixel_sizes:
        size_z_um = size_z * pixel_sizes["Z"][0]
        print(f"  Z: {size_z_um:.2f} {pixel_sizes['Z'][1]}")
else:
    print("\n⚠️ Could not compute physical field of view — missing scaling or dimensions.")

🧪 Objective Metadata:
  ID: Objective:1
  Name: Plan-Apochromat 63x/1.40 Oil DIC M27
  Magnification: 63x
  Numerical Aperture (NA): 1.4
  Working Distance: 193 µm
  Pupil Geometry: Circular
  Immersion Type: Oil
  Immersion Refractive Index: 1.518
  Manufacturer Model: Plan-Apochromat 63x/1.40 Oil DIC M27
🧬 Pixel Size Metadata:
  X: 1.315e-07 µm
  Y: 1.315e-07 µm
  Z: 2.000e-07 µm


In [1]:
from tifffile import TiffFile

tif = TiffFile("test_asia.tif")

ij_meta = tif.imagej_metadata
if ij_meta:
    print("🧬 ImageJ Metadata:")
    for k, v in ij_meta.items():
        print(f"  {k}: {v}")

    # Dimensions
    size_z = ij_meta.get("slices", 1)
    size_t = ij_meta.get("frames", 1)
    size_c = ij_meta.get("channels", 1)
    size_y, size_x = tif.pages[0].shape

    print("\n📏 Image Dimensions (pixels):")
    print(f"  X: {size_x}")
    print(f"  Y: {size_y}")
    print(f"  Z: {size_z}")
    print(f"  C: {size_c}")
    print(f"  T: {size_t}")

    # Physical scaling (if available)
    pixel_width = ij_meta.get("x_resolution") or ij_meta.get("pixel_width")
    pixel_height = ij_meta.get("y_resolution") or ij_meta.get("pixel_height")
    voxel_depth = ij_meta.get("spacing") or ij_meta.get("z_spacing")

    if pixel_width:
        print("\n🧬 Pixel Size (if available):")
        print(f"  X: {pixel_width}")
        print(f"  Y: {pixel_height}")
        print(f"  Z: {voxel_depth}")
else:
    print("⚠️ No ImageJ metadata found.")

🧬 ImageJ Metadata:
  ImageJ: 1.54p
  images: 78
  slices: 78
  unit: micron
  spacing: 0.5
  loop: False
  min: 0.0
  max: 65535.0
  Labels: ['c:3/4 z:1/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:2/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:3/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:4/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:5/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:6/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:7/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:8/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:9/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:10/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:11/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:12/78 - 40X_F50M50_50M_Static_down_C3_001.nd2 (series 1)', 'c:3/4 z:13/78 - 40X_F50M50_50M_Static_down_C3_001.

In [2]:
from tifffile import TiffFile
import xml.etree.ElementTree as ET
import numpy as np

tif_path = "test_asia.tif"

with TiffFile(tif_path) as tif:
    ome_metadata = tif.ome_metadata
    ij_meta = tif.imagej_metadata

    # --- If it's an OME-TIFF ---
    if ome_metadata:
        print("📦 Detected OME-TIFF")
        root = ET.fromstring(ome_metadata)
        ns = {"ome": "http://www.openmicroscopy.org/Schemas/OME/2016-06"}
        pixels = root.find(".//ome:Pixels", ns)

        if pixels is not None:
            size_x = int(pixels.attrib["SizeX"])
            size_y = int(pixels.attrib["SizeY"])
            size_z = int(pixels.attrib.get("SizeZ", 1))
            size_c = int(pixels.attrib.get("SizeC", 1))
            size_t = int(pixels.attrib.get("SizeT", 1))

            # pixel sizes (often in µm)
            phys_x = float(pixels.attrib.get("PhysicalSizeX", "nan"))
            phys_y = float(pixels.attrib.get("PhysicalSizeY", "nan"))
            phys_z = float(pixels.attrib.get("PhysicalSizeZ", "nan"))
            unit = pixels.attrib.get("PhysicalSizeXUnit", "µm")

            print("\n📏 Image Dimensions (pixels):")
            print(f"  X: {size_x}")
            print(f"  Y: {size_y}")
            print(f"  Z: {size_z}")

            print("\n🧬 Pixel Size:")
            print(f"  X: {phys_x} {unit}/px")
            print(f"  Y: {phys_y} {unit}/px")
            print(f"  Z: {phys_z} {unit}/px")

            # Compute total physical size
            if not np.isnan(phys_x):
                total_x = size_x * phys_x
                total_y = size_y * phys_y
                total_z = size_z * phys_z
                print("\n🌍 Physical Field of View:")
                print(f"  X: {total_x:.2f} {unit}")
                print(f"  Y: {total_y:.2f} {unit}")
                print(f"  Z: {total_z:.2f} {unit}")
            else:
                print("\n⚠️ Pixel size not found in metadata.")
        else:
            print("⚠️ No <Pixels> metadata found in OME-XML.")

    # --- If it’s an ImageJ TIFF ---
    elif ij_meta:
        print("📦 Detected ImageJ TIFF")
        size_z = ij_meta.get("slices", 1)
        size_t = ij_meta.get("frames", 1)
        size_c = ij_meta.get("channels", 1)
        size_y, size_x = tif.pages[0].shape

        pixel_width = ij_meta.get("pixel_width")
        pixel_height = ij_meta.get("pixel_height")
        voxel_depth = ij_meta.get("spacing")
        unit = ij_meta.get("unit", "µm")

        print("\n📏 Image Dimensions (pixels):")
        print(f"  X: {size_x}")
        print(f"  Y: {size_y}")
        print(f"  Z: {size_z}")

        print("\n🧬 Pixel Size:")
        print(f"  X: {pixel_width} {unit}/px")
        print(f"  Y: {pixel_height} {unit}/px")
        print(f"  Z: {voxel_depth} {unit}/px")

        if pixel_width:
            total_x = size_x * pixel_width
            total_y = size_y * pixel_height
            total_z = size_z * voxel_depth if voxel_depth else None

            print("\n🌍 Physical Field of View:")
            print(f"  X: {total_x:.2f} {unit}")
            print(f"  Y: {total_y:.2f} {unit}")
            if total_z:
                print(f"  Z: {total_z:.2f} {unit}")
    else:
        # --- Plain TIFF fallback ---
        print("⚠️ No OME or ImageJ metadata found.")
        shape = tif.asarray().shape
        print("📏 Image Dimensions (pixels):", shape)

📦 Detected ImageJ TIFF

📏 Image Dimensions (pixels):
  X: 1360
  Y: 1360
  Z: 78

🧬 Pixel Size:
  X: None micron/px
  Y: None micron/px
  Z: 0.5 micron/px


In [8]:
from tifffile import TiffFile
import pandas as pd
import numpy as np

def get_tiff_metadata(tiff_path):
    """
    Extract pixel and physical dimensions (in microns) from an ImageJ or OME-TIFF file.
    Works when ImageJ shows both pixel and micron sizes.
    """
    with TiffFile(tiff_path) as tif:
        ij_meta = tif.imagej_metadata
        ome_meta = tif.ome_metadata

        # --- Basic pixel dimensions ---
        size_y, size_x = tif.pages[0].shape
        n_pages = len(tif.pages)

        size_z = 1
        size_c = 1
        size_t = 1
        if ij_meta:
            size_z = ij_meta.get("slices", 1)
            size_c = ij_meta.get("channels", 1)
            size_t = ij_meta.get("frames", 1)

        # --- Default pixel sizes (µm/px) ---
        px_x = px_y = px_z = np.nan

        # --- Case 1: ImageJ metadata ---
        if ij_meta:
            px_x = ij_meta.get("pixel_width") or ij_meta.get("x_resolution")
            px_y = ij_meta.get("pixel_height") or ij_meta.get("y_resolution")
            px_z = ij_meta.get("spacing") or ij_meta.get("z_spacing")
            unit = ij_meta.get("unit", "µm")

        # --- Case 2: OME-TIFF fallback ---
        elif ome_meta:
            import xml.etree.ElementTree as ET
            root = ET.fromstring(ome_meta)
            ns = {"ome": "http://www.openmicroscopy.org/Schemas/OME/2016-06"}
            pixels = root.find(".//ome:Pixels", ns)
            if pixels is not None:
                px_x = float(pixels.attrib.get("PhysicalSizeX", np.nan))
                px_y = float(pixels.attrib.get("PhysicalSizeY", np.nan))
                px_z = float(pixels.attrib.get("PhysicalSizeZ", np.nan))
                unit = pixels.attrib.get("PhysicalSizeXUnit", "µm")
            else:
                unit = "µm"
        else:
            unit = "µm"

        # --- Compute total physical field of view ---
        fx = size_x * px_x if px_x else np.nan
        fy = size_y * px_y if px_y else np.nan
        fz = size_z * px_z if px_z else np.nan

        # --- Return as DataFrame ---
        df = pd.DataFrame({
            "Axis": ["X", "Y", "Z"],
            "Pixels": [size_x, size_y, size_z],
            f"Voxel size ({unit}/px)": [px_x, px_y, px_z],
            f"Field of View ({unit})": [fx, fy, fz],
        })

        return df

#Example usage in Jupyter:
df = get_tiff_metadata("test_asia.tif")
display(df)


,Axis,Pixels,Voxel size (micron/px),Field of View (micron)
0,X,1360,NaN,NaN
1,Y,1360,NaN,NaN
2,Z,78,0.5,39.0
